# 📝 Instrucciones de la tarea

En esta práctica trabajarás con una implementación propia de un modelo Transformer en PyTorch para traducir frases del español al inglés.

## ✅ Objetivos:
- Cargar un mini dataset de frases paralelas.
- Preparar los vocabularios de entrada y salida.
- Codificar las frases a tensores con padding.
- Entrenar el modelo Transformer durante varias épocas.
- Realizar inferencias: traducir frases nuevas.

## 🧪 Qué debes entregar:
- El notebook completo con todas las celdas ejecutadas.
- Al menos 2 ejemplos de frases nuevas traducidas.
- Un comentario breve (2-3 líneas) explicando cómo crees que el modelo ha aprendido.

## 💡 Sugerencias:
- Puedes añadir nuevas frases al dataset si quieres mejorar el entrenamiento.
- Si el modelo no aprende bien, prueba con más épocas o un tamaño de embedding mayor.
- Añade un `print` intermedio para ver cómo evoluciona la salida del modelo durante el entrenamiento.

# 🧪 Mini Proyecto: Traducción Español → Inglés con Transformer en PyTorch

Este notebook utiliza un modelo Transformer definido manualmente para traducir frases cortas del español al inglés.

## Dataset usado
Archivo: `mini_trad_es_en.tsv`
Formato: frase_en_español <tab> frase_en_ingles

In [46]:
# 📥 Cargar datos y preparar vocabularios
from transformerPytorch import *
import torch
from collections import defaultdict

# Cargar pares
data = []
with open('mini_trad_es_en.tsv', encoding='utf-8') as f:
    for line in f:
        if line.strip() and not line.startswith('#'):
            src, tgt = line.strip().split('\t')
            data.append((src.lower().split(), tgt.lower().split()))

# Crear vocabularios
def build_vocab(sequences):
    vocab = {'<pad>':0, '<sos>':1, '<eos>':2, '<unk>':3}
    idx = 4
    for seq in sequences:
        for token in seq:
            if token not in vocab:
                vocab[token] = idx
                idx += 1
    return vocab

src_vocab = build_vocab([src for src, _ in data])
tgt_vocab = build_vocab([tgt for _, tgt in data])
inv_tgt_vocab = {v:k for k,v in tgt_vocab.items()}

## 🔄 Codificación de secuencias y padding

In [47]:
# Convertir frases a tensores con padding
def encode(seq, vocab, max_len):
    ids = [vocab.get(tok, vocab['<unk>']) for tok in seq]
    ids = [vocab['<sos>']] + ids + [vocab['<eos>']]
    if len(ids) < max_len:
        ids += [vocab['<pad>']] * (max_len - len(ids))
    return torch.tensor(ids[:max_len])

max_src_len = max(len(s) for s, _ in data) + 2
max_tgt_len = max(len(t) for _, t in data) + 2

pairs = [(encode(s, src_vocab, max_src_len), encode(t, tgt_vocab, max_tgt_len)) for s,t in data]

# Crear dataloader
from torch.utils.data import DataLoader
dataloader = DataLoader(pairs, batch_size=2, shuffle=True)

## 🏋️ Entrenamiento del Transformer

In [48]:
# 🏋️ Entrenamiento del modelo Transformer con el dataset cargado
# Asegúrate de haber importado y definido tu clase Transformer previamente
import torch.nn as nn
import torch.optim as optim

# Hiperparámetros básicos
src_vocab_size = len(src_vocab)
tgt_vocab_size = len(tgt_vocab)
embed_size = 32
nhead = 2
num_layers = 2

# Instanciar modelo
# Debes reemplazar esto por tu clase Transformer
model = Transformer(src_vocab_size, tgt_vocab_size, embed_size, nhead, num_layers, d_ff=2, max_seq_length=100, dropout=0.1)
criterion = nn.CrossEntropyLoss(ignore_index=tgt_vocab['<pad>'])
optimizer = optim.Adam(model.parameters(), lr=0.001)

model.train()
for epoch in range(400):
    total_loss = 0
    for src, tgt in dataloader:
        tgt_input = tgt[:, :-1]
        tgt_output = tgt[:, 1:].reshape(-1)
        
        logits = model(src, tgt_input)
        loss = criterion(logits.reshape(-1, logits.size(-1)), tgt_output)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}, Loss: {total_loss:.4f}")

Epoch 1, Loss: 55.4395
Epoch 2, Loss: 49.4676
Epoch 3, Loss: 46.6425
Epoch 4, Loss: 43.2418
Epoch 5, Loss: 40.6045
Epoch 6, Loss: 38.3313
Epoch 7, Loss: 35.4687
Epoch 8, Loss: 33.1558
Epoch 9, Loss: 31.5896
Epoch 10, Loss: 29.2673
Epoch 11, Loss: 27.0976
Epoch 12, Loss: 24.8721
Epoch 13, Loss: 23.1142
Epoch 14, Loss: 21.6235
Epoch 15, Loss: 19.6672
Epoch 16, Loss: 18.1348
Epoch 17, Loss: 17.1786
Epoch 18, Loss: 16.1073
Epoch 19, Loss: 15.9318
Epoch 20, Loss: 14.1043
Epoch 21, Loss: 12.8325
Epoch 22, Loss: 12.7902
Epoch 23, Loss: 11.5745
Epoch 24, Loss: 11.2410
Epoch 25, Loss: 10.1322
Epoch 26, Loss: 9.0376
Epoch 27, Loss: 8.9291
Epoch 28, Loss: 7.9248
Epoch 29, Loss: 8.1492
Epoch 30, Loss: 7.4536
Epoch 31, Loss: 6.7229
Epoch 32, Loss: 6.9958
Epoch 33, Loss: 6.5148
Epoch 34, Loss: 6.1585
Epoch 35, Loss: 5.5643
Epoch 36, Loss: 5.4221
Epoch 37, Loss: 5.2351
Epoch 38, Loss: 4.5590
Epoch 39, Loss: 4.2359
Epoch 40, Loss: 4.2571
Epoch 41, Loss: 3.7944
Epoch 42, Loss: 4.1664
Epoch 43, Loss: 3.

## 🔍 Prueba de inferencia

In [51]:
# 🔍 Inferencia: generar traducción para una frase nueva
def translate_sentence(sentence):
    model.eval()
    tokens = sentence.lower().split()
    src_tensor = encode(tokens, src_vocab, max_src_len).unsqueeze(0)
    tgt_indices = [tgt_vocab['<sos>']]

    for _ in range(max_tgt_len):
        tgt_tensor = torch.tensor(tgt_indices).unsqueeze(0)
        with torch.no_grad():
            output = model(src_tensor, tgt_tensor)
        next_token = output[0, -1].argmax().item()
        if next_token == tgt_vocab['<eos>']:
            break
        tgt_indices.append(next_token)

    translated = [inv_tgt_vocab[idx] for idx in tgt_indices[1:]]
    return ' '.join(translated)

# Prueba
print(translate_sentence("hola"))
print(translate_sentence("adiós"))
print(translate_sentence("¿cómo estás?"))
print(translate_sentence("me gusta el chocolate"))
print(translate_sentence("good night"))
print(translate_sentence("¿qué hora es?"))
print(translate_sentence("tengo hambre"))
print(translate_sentence("me llamo ana"))
print(translate_sentence("gracias"))
print(translate_sentence("buenos días"))
print(translate_sentence("¿dónde está el baño?"))
print(translate_sentence("me gusta la música"))
print(translate_sentence("David come chocolate"))

hello
goodbye
how are you?
i like ice cream
thank you
what time is it?
i am hungry
my name is ana
thank you
good morning
thank you
i like ice cream
ana eats chocolate
